# Inspect features and retain per-fold selections

This companion loads pinned Premier League 2023/24–2024/25 publications, inspects
training rows, and applies selected names to test-feature distributions. It runs
descriptive studies only. Corners are a compact API example, not a chosen pilot
feature recipe. Use the existing `misc314_py314` kernel with reporting dependencies.

See the [guide](../docs/analytics/reporting.md) and
[complete reference](../docs/analytics/reporting_reference.md).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from xdiyo_analytics.data import load_seasons, select_stats
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import IsHome, Lag, Stat, evaluate_features
from xdiyo_analytics.labels import MatchTotal, create_labels
from xdiyo_analytics.datasets import assemble_dataset
from xdiyo_analytics.splits import TemporalSplit, create_split_plan
from xdiyo_analytics.analysis import PreTrainingAnalysis
from xdiyo_analytics.reporting import (
    FeatureDistributionReporter, CorrelationAnalysis, FeatureTimeline,
    TopKCorrelationSelector, FeatureSelector, FeatureSelection, vote_selections,
    Artifact, StudyResult,
)

root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')
loaded = load_seasons(
    root / 'data/xDiyo_data', ['23_24', '24_25'], leagues='Premier_League',
    tables=['matches', 'statistics'], verify_hashes=True,
    record_dir=root / 'experiment/initial_population/selections',
)
history = build_team_history(select_stats(
    loaded, stats=[('ALL', 'Match overview', 'cornerKicks')],
))
corners = Stat('ALL', 'Match overview', 'cornerKicks')
features = evaluate_features(history, {
    'venue': IsHome(), 'previous': Lag(corners), 'previous_2': Lag(corners, 2),
}, keyed=True)
labels = create_labels(history, {'corners': MatchTotal(corners)})
dataset = assemble_dataset(features, labels['corners'], layout='match')
plan = create_split_plan(dataset, TemporalSplit(
    20, test_size=8, step=20, allow_partial_test=True,
))


In [2]:
report = PreTrainingAnalysis({
    'training associations': CorrelationAnalysis(
        type='per_fold', partition='train', methods=['pearson', 'spearman'],
    ),
    'training selection': TopKCorrelationSelector(
        type='per_fold', partition='train', k=2, source='training associations',
    ),
    'selected test features': FeatureDistributionReporter(
        type='per_fold', partition='test', features_from='training selection', bins=8,
    ),
}, title='Notebook pre-training inspection').run(
    dataset, split_plan=plan, fold_ids=[0, 1],
)
pd.DataFrame([{'study': r.name, 'fold': r.fold_id, 'partition': r.partition,
               'rows': len(r.row_positions), 'matches': r.n_matches}
              for r in report.studies])

,study,fold,partition,rows,matches
0,training associations,0,train,198,198
1,training associations,1,train,400,400
2,training selection,0,train,198,198
3,training selection,1,train,400,400
4,selected test features,0,test,80,80
5,selected test features,1,test,80,80


## Keep the selection with its fold

Selection scores come from training rows. Applying names to a test distribution
does not use test labels to choose features. Later reporters need explicit
`features_from` to use a selection; the source dataset stays intact.

In [3]:
pd.DataFrame([
    {'fold': fold, 'columns': selection.columns}
    for fold, selection in report.selections['training selection'].items()
])

,fold,columns
0,0,"(away::previous_2, home::previous)"
1,1,"(away::previous_2, home::previous_2)"


## Open the interactive report

The next cells export standalone HTML, construct a native `IPython.display.IFrame`,
and display the report by leaving `report` as the final expression. Alternatively,
display `inline_view` or call `report.show(height=700)`. These actions reuse computed
results. Use the sidebar, fold controls and chart/data panels.

The notebook frontend must trust and permit HTML/JavaScript. Bundled Plotly makes
the embedded output larger. Use the standalone export in a restrictive viewer.

In [4]:
from IPython.display import IFrame
output_path = root / 'notebooks/outputs/reporting_quickstart.html'
_ = report.to_html(output_path)
inline_view = report.to_notebook(height=700)
assert isinstance(inline_view, IFrame)
output_path.name

'reporting_quickstart.html'

In [5]:
report

AnalysisReport(studies=[StudyRun(name='training associations', type='per_fold', partition='train', fold_id=0, layout='match', row_positions=array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181,
       182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194,
       195, 196, 197]), n_matches=198, result=StudyResult(title='Feature–target associations', artifacts=[Artifact(kind='leaderboard', data=            feature  pooled_magnitude  n_coefficients  corners · pearson  \
0  away::previous_2          0.286282               2          -0.143829   
1    home::previous          0.068701               2           0.031759   
2  home::previous_2          0.049573               2           0.030166   
3    away::previous          0.027252               2           0.012926   
4       home::venue               NaN               0                NaN   
5       away::venue               NaN               0                NaN   

   corners · pearson · percentile  corners · spearman  \
0                           100.0           -0.142453   
1                            75.0            0.036943   
2                            50.0            0.019407   
3                            25.0           -0.014327   
4                             NaN                 NaN   
5                             NaN                 NaN   

   corners · spearman · percentile  
0                            100.0  
1                             75.0  
2                             50.0  
3                             25.0  
4                              NaN  
5                              NaN  , title='Association leaderboard', options={'coefficient_columns': ['corners · pearson', 'corners · spearman'], 'percentile_columns': ['corners · pearson · percentile', 'corners · spearman · percentile']})], tables={'leaderboard':             feature  pooled_magnitude  n_coefficients  corners · pearson  \
0  away::previous_2          0.286282               2          -0.143829   
1    home::previous          0.068701               2           0.031759   
2  home::previous_2          0.049573               2           0.030166   
3    away::previous          0.027252               2           0.012926   
4       home::venue               NaN               0                NaN   
5       away::venue               NaN               0                NaN   

   corners · pearson · percentile  corners · spearman  \
0                           100.0           -0.142453   
1                            75.0            0.036943   
2                            50.0            0.019407   
3                            25.0           -0.014327   
4                             NaN                 NaN   
5                             NaN                 NaN   

   corners · spearman · percentile  
0                            100.0  
1                             75.0  
2                             50.0  
3                             25.0  
4                              NaN  
5                              NaN  , 'coefficients':          

## Interpretation

Every supplied row has equal statistical weight. These match-layout examples use
one row per match; team-match data retains two observations per complete match.
Missing values remain explicit, and undefined associations remain missing.

Overall reports pool unique selected rows. Fold consensus is an explicit,
separate selector mode; it needs an untouched outer test for performance claims.
No predictive model was fitted here. Fold-aware rolling/rating state reconstruction
and post-training metrics remain later layers.

[Verification coverage](../docs/analytics/reporting_documentation_checklist.md) ·
[Verification evidence](../docs/analytics/reporting_check.json)